In [5]:
# Multi-Agent CEO Intelligence System (Odoo Enterprise)
# Hardened. Defensive. CEO-safe.

"""
WHAT THIS IS
------------
A read-only, multi-agent executive intelligence layer on top of Odoo.
It NEVER crashes on empty data.
It NEVER assumes permissions.
It ONLY escalates contradictions, risks, and leverage.
"""

# ================================================================
# IMPORTS
# ================================================================
import os
import xmlrpc.client
import pandas as pd
from typing import Dict, Any
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# ================================================================
# CONFIG
# ================================================================
load_dotenv()

# Initialize with your credentials
# Initialize with your credentials
ODOO_URL = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
ODOO_DB = os.getenv('ODOO_DB', 'vendyltd')
ODOO_USER = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
ODOO_PASSWORD = os.getenv('ODOO_PASSWORD', '205958c9bb841b4bef7e6da4a3afb5b5029cd6ae')

LLM_MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# ================================================================
# ODOO CLIENT (READ ONLY)
# ================================================================
class OdooClient:
    def __init__(self):
        common = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/common")
        self.uid = common.authenticate(ODOO_DB, ODOO_USER, ODOO_PASSWORD, {})
        self.models = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/object")

    def read(self, model, domain=None, fields=None):
        return self.models.execute_kw(
            ODOO_DB,
            self.uid,
            ODOO_PASSWORD,
            model,
            'search_read',
            [domain or []],
            {'fields': fields or []}
        )

odoo = OdooClient()

# ================================================================
# SAFETY UTILITIES (NON-NEGOTIABLE)
# ================================================================

def safe_df(records, required_cols):
    """
    Guarantees a DataFrame with required columns.
    Survives empty tables and missing permissions.
    """
    if not records:
        return pd.DataFrame(columns=required_cols)

    df = pd.DataFrame(records)

    for col in required_cols:
        if col not in df.columns:
            df[col] = 0

    return df


# ================================================================
# AGENT 1 — SALES INTELLIGENCE
# ================================================================

def sales_agent() -> Dict[str, Any]:
    orders = odoo.read(
        'sale.order',
        fields=['amount_total', 'state']
    )

    df = safe_df(orders, ['amount_total', 'state'])

    return {
        "total_sales": round(df['amount_total'].astype(float).sum(), 2),
        "draft_orders": int((df['state'] == 'draft').sum()),
        "order_count": int(len(df))
    }


# ================================================================
# AGENT 2 — DELIVERY & OPERATIONS
# ================================================================

def delivery_agent() -> Dict[str, Any]:
    pickings = odoo.read(
        'stock.picking',
        fields=['state']
    )

    df = safe_df(pickings, ['state'])

    total = len(df)
    done = int((df['state'] == 'done').sum())

    return {
        "pending_deliveries": int(total - done),
        "delivery_completion_rate": round((done / total) * 100, 2) if total else 0
    }


# ================================================================
# AGENT 3 — INVENTORY & STOCK RISK
# ================================================================

def inventory_agent() -> Dict[str, Any]:
    quants = odoo.read(
        'stock.quant',
        fields=['quantity']
    )

    df = safe_df(quants, ['quantity'])

    return {
        "total_stock_units": float(df['quantity'].astype(float).sum()),
        "zero_or_negative_stock": int((df['quantity'].astype(float) <= 0).sum())
    }


# ================================================================
# AGENT 4 — ACCOUNTING & CASH REALITY
# ================================================================

def accounting_agent() -> Dict[str, Any]:
    moves = odoo.read(
        'account.move',
        fields=['amount_total', 'move_type', 'state']
    )

    df = safe_df(moves, ['amount_total', 'move_type', 'state'])

    income = df[(df['move_type'] == 'out_invoice') & (df['state'] == 'posted')]['amount_total'].astype(float).sum()
    expenses = df[(df['move_type'] == 'in_invoice') & (df['state'] == 'posted')]['amount_total'].astype(float).sum()

    return {
        "revenue": round(income, 2),
        "expenses": round(expenses, 2),
        "profit": round(income - expenses, 2)
    }


# ================================================================
# AGENT 5 — CFO CASH-FLOW vs PROFIT vs AR/AP (HARD TRUTH)
# ================================================================

def cfo_cashflow_agent(human_input: Dict[str, Any] | None = None) -> Dict[str, Any]:
    """
    CFO-grade financial reality check.
    Separates accounting profit from cash truth.
    Allows human overrides (date ranges, thresholds).
    """

    date_from = human_input.get('date_from') if human_input else None
    date_to = human_input.get('date_to') if human_input else None

    domain = []
    if date_from:
        domain.append(('date', '>=', date_from))
    if date_to:
        domain.append(('date', '<=', date_to))

    moves = odoo.read(
        'account.move',
        domain=domain,
        fields=['amount_total', 'move_type', 'state', 'payment_state']
    )

    df = safe_df(moves, ['amount_total', 'move_type', 'state', 'payment_state'])
    df['amount_total'] = df['amount_total'].astype(float)

    revenue = df[(df['move_type'] == 'out_invoice') & (df['state'] == 'posted')]['amount_total'].sum()
    expenses = df[(df['move_type'] == 'in_invoice') & (df['state'] == 'posted')]['amount_total'].sum()

    cash_in = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()
    cash_out = df[(df['move_type'] == 'in_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()

    ar = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] != 'paid')]['amount_total'].sum()
    ap = df[(df['move_type'] == 'in_invoice') & (df['payment_state'] != 'paid')]['amount_total'].sum()

    return {
        "profit": round(revenue - expenses, 2),
        "cash_flow": round(cash_in - cash_out, 2),
        "accounts_receivable": round(ar, 2),
        "accounts_payable": round(ap, 2),
        "profit_vs_cash_gap": round((revenue - expenses) - (cash_in - cash_out), 2)
    }


# ================================================================
# AGENT 6 — CONTRADICTION DETECTOR (CEO GOLD)
# ================================================================

def contradiction_agent(state: Dict[str, Any]) -> Dict[str, Any]:
    flags = []

    if state['sales']['total_sales'] > 0 and state['accounting']['revenue'] == 0:
        flags.append("Sales recorded but no invoices posted")

    if state['delivery']['delivery_completion_rate'] > 80 and state['accounting']['revenue'] == 0:
        flags.append("Goods delivered but cash not invoiced")

    if state['inventory']['total_stock_units'] > 0 and state['sales']['order_count'] == 0:
        flags.append("Stock piling without demand")

    if state['cfo']['profit'] > 0 and state['cfo']['cash_flow'] < 0:
        flags.append("Profitable on paper but bleeding cash")

    if state['cfo']['accounts_receivable'] > state['sales']['total_sales'] * 0.6:
        flags.append("Dangerous AR concentration — cash collection risk")

    return {
        "contradictions": flags or ["No critical contradictions detected"]
    }


# ================================================================


def contradiction_agent(state: Dict[str, Any]) -> Dict[str, Any]:
    flags = []

    if state['sales']['total_sales'] > 0 and state['accounting']['revenue'] == 0:
        flags.append("Sales recorded but no invoices posted")

    if state['delivery']['delivery_completion_rate'] > 80 and state['accounting']['revenue'] == 0:
        flags.append("Goods delivered but cash not invoiced")

    if state['inventory']['total_stock_units'] > 0 and state['sales']['order_count'] == 0:
        flags.append("Stock piling without demand")

    if state['accounting']['profit'] > 0 and state['inventory']['zero_or_negative_stock'] > 0:
        flags.append("Profitable on paper but stock risk exists")

    return {
        "contradictions": flags or ["No critical contradictions detected"]
    }


# ================================================================
# CEO SYNTHESIS AGENT (ONLY VOICE)
# ================================================================

def ceo_agent(state: Dict[str, Any]) -> Dict[str, Any]:
    prompt = f"""
You are the CEO.
No operational excuses.
Focus on truth, risk, and leverage.

SYSTEM SNAPSHOT:
{state}

Deliver:
1. One-line company health verdict
2. 3 hard insights
3. 3 actions for next 30 days
4. 1 existential risk
"""

    response = llm.invoke(prompt)
    return {"CEO_REPORT": response.content}


# ================================================================
# LANGGRAPH ORCHESTRATION
# ================================================================

graph = StateGraph(dict)

def collect_agents(state):
    # ---------------- HUMAN IN THE LOOP ----------------
    # CEO / CFO can inject constraints without touching code
    human_input = state.get('human_input', {})

    snapshot = {
        "sales": sales_agent(),
        "delivery": delivery_agent(),
        "inventory": inventory_agent(),
        "accounting": accounting_agent(),
        "cfo": cfo_cashflow_agent(human_input)
    }

    snapshot.update(contradiction_agent(snapshot))
    return snapshot


graph.add_node("collect_agents", collect_agents)
graph.add_node("ceo_agent", ceo_agent)

graph.set_entry_point("collect_agents")
graph.add_edge("collect_agents", "ceo_agent")
graph.add_edge("ceo_agent", END)

app = graph.compile()

# ================================================================
# ENTRY POINT
# ================================================================
if __name__ == "__main__":
    print("Multi-Agent CEO System Online")
    result = app.invoke({
        "human_input": {
            # Optional CEO / CFO overrides
            # "date_from": "2025-01-01",
            # "date_to": "2025-03-31"
        }
    })
    print("\n--- CEO EXECUTIVE BRIEF ---\n")
    print(result['CEO_REPORT'])


Multi-Agent CEO System Online

--- CEO EXECUTIVE BRIEF ---

1. **Company Health Verdict:** The company is profitable on paper but faces significant operational risks due to zero inventory and pending deliveries.

2. **Hard Insights:**
   - **Inventory Crisis:** With 11 stock units at zero or negative levels, the company is unable to fulfill orders, jeopardizing customer satisfaction and future sales.
   - **Cash Flow Discrepancy:** Despite showing a profit of $293,447.12, the cash flow is currently at $0.00, indicating potential liquidity issues if accounts receivable are not collected promptly.
   - **Delivery Challenges:** A delivery completion rate of only 60% suggests inefficiencies in logistics, which could lead to customer dissatisfaction and lost revenue opportunities.

3. **Actions for Next 30 Days:**
   - **Inventory Replenishment:** Immediately source and order necessary stock to address the zero inventory issue and ensure fulfillment of pending deliveries.
   - **Cash Flow M

In [8]:
from IPython.display import display, Markdown
display(Markdown(open('CEO_Report_INDIVIDUAL_DEEPDIVE_20251217_055731.md', encoding='utf-8').read()))


# 📑 CEO Executive Brief: Individual Deepdive
**Date:** 2025-12-17 | **Period:** None to None
**Scope:** Company

---

## 1. 📢 Executive Summary
**Employee Deep Dive: None**


**Period Summary:**
- Worked Hours: 0.0
- Total Payable: $0.00

---

## 2. 📊 Financial Breakdown
| Metric | Value |
| :--- | :--- |
| **Base Salary Liability** | $0.00 |
| **(+) Estimated Overtime** | $0.00 |
| **(-) Performance/Late Deductions** | $0.00 |
| **= TOTAL PROJECTED PAYROLL** | **$0.00** |
| **Total Worked Hours** | **0.0 hours** |

---

## 3. 📋 Detailed Breakdown
No active contracts found for the specified period.

---
*Generated by Odoo CEO Agent (Pro Edition)* | *Date: 2025-12-17 05:57:31*
